# P&C Loss Reserving: Chain Ladder and Bornhuetter-Ferguson Reserve Estimate

This notebook builds a property and casualty loss reserving analysis in two layers.

Layer 1 is a transparent manual analysis using an in-notebook cumulative paid loss triangle. It calculates age-to-age loss development factors, cumulative development factors, Chain Ladder ultimate losses, IBNR, and Bornhuetter-Ferguson reserve indications.

Layer 2 uses the `chainladder` package to load the RAA sample triangle directly from the library and compare a package-based Chain Ladder result with the manual calculation framework.

The workflow follows the core structure of a reserving review: organize triangle data, select development factors, estimate ultimate claims, compare methods, assess reasonableness, and communicate reserve indications with limitations.


## 1. Imports, Setup, and Helper Functions


In [ ]:
# Import the core analysis libraries, set the project output folder, and define
# reusable reserving helper functions used in both Layer 1 and Layer 2.

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Check that the actuarial chainladder package is available before running
# Layer 2. Failing here gives a clearer error than failing later in the analysis.
try:
    import chainladder as cl
    print(f"chainladder {cl.__version__}")
except ImportError as exc:
    raise ImportError(
        "chainladder is required for the Layer 2 package analysis. "
        "Install chainladder in the notebook kernel environment and rerun."
    ) from exc


def find_project_root() -> Path:
    """Find the project root so output files are written to a stable folder."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "outputs").exists() and (candidate / "projects").exists():
            return candidate
    return Path.cwd()


def calculate_ldf(triangle_df: pd.DataFrame) -> pd.Series:
    """Calculate selected age-to-age loss development factors.

    For each adjacent pair of development ages, use only rows where both
    maturities are observed. This avoids using incomplete accident years in the
    ratio and matches a common volume-weighted Chain Ladder selection.
    """
    ldfs = {}

    for i in range(len(triangle_df.columns) - 1):
        current_dev = triangle_df.columns[i]
        next_dev = triangle_df.columns[i + 1]

        # Drop rows where the current or next development value is missing.
        # The ratio is volume-weighted: total next age / total current age.
        valid = triangle_df[[current_dev, next_dev]].dropna()
        ldfs[f"{current_dev}-{next_dev}"] = valid[next_dev].sum() / valid[current_dev].sum()

    return pd.Series(ldfs, name="selected_ldf")


def calculate_cdf(ldfs: pd.Series) -> pd.Series:
    """Convert age-to-age LDFs into CDFs to ultimate.

    A CDF is the product of all future LDFs from the current maturity onward.
    For example, the 24-month CDF equals LDF 24-36 x LDF 36-48 x LDF 48-60.
    """
    cdf_values = [np.prod(ldfs.iloc[i:]) for i in range(len(ldfs))]
    labels = [label.split("-")[0] for label in ldfs.index]
    return pd.Series(cdf_values, index=labels, name="cdf_to_ultimate")


def latest_diagonal(triangle_df: pd.DataFrame) -> pd.DataFrame:
    """Extract the latest observed maturity and paid amount for each accident year."""
    rows = []

    for accident_year, row in triangle_df.iterrows():
        observed = row.dropna()
        rows.append({
            "accident_year": accident_year,
            "latest_dev": int(observed.index.max()),
            "latest_paid": observed.iloc[-1],
        })

    return pd.DataFrame(rows).set_index("accident_year")


def chain_ladder_reserve(triangle_df: pd.DataFrame) -> pd.DataFrame:
    """Estimate ultimate loss and IBNR using the Chain Ladder method."""
    selected_ldfs = calculate_ldf(triangle_df)
    selected_cdfs = calculate_cdf(selected_ldfs)
    latest = latest_diagonal(triangle_df)
    rows = []

    for accident_year, row in latest.iterrows():
        latest_dev = int(row["latest_dev"])
        latest_paid = row["latest_paid"]

        # If an accident year is already at the final maturity, no future
        # development is applied, so the CDF defaults to 1.0.
        cdf = selected_cdfs.get(str(latest_dev), 1.0)
        ultimate_loss = latest_paid * cdf

        rows.append({
            "accident_year": accident_year,
            "latest_dev": latest_dev,
            "latest_paid": latest_paid,
            "cdf_to_ultimate": cdf,
            "cl_ultimate_loss": ultimate_loss,
            "cl_ibnr": ultimate_loss - latest_paid,
        })

    return pd.DataFrame(rows).set_index("accident_year")


def bornhuetter_ferguson_reserve(
    cl_result_df: pd.DataFrame,
    earned_premium: dict[int, float],
    expected_loss_ratio: float,
) -> pd.DataFrame:
    """Estimate IBNR with the Bornhuetter-Ferguson method.

    BF reserves use an a priori expected loss estimate, then apply only the
    unreported percentage implied by the development pattern. This is often
    more stable than pure Chain Ladder for immature accident years.
    """
    bf = cl_result_df.copy()

    bf["earned_premium"] = bf.index.map(earned_premium)
    bf["expected_loss"] = bf["earned_premium"] * expected_loss_ratio
    bf["percent_reported"] = 1 / bf["cdf_to_ultimate"]
    bf["unreported_percent"] = 1 - bf["percent_reported"]
    bf["bf_ibnr"] = bf["expected_loss"] * bf["unreported_percent"]
    bf["bf_ultimate_loss"] = bf["latest_paid"] + bf["bf_ibnr"]

    return bf


PROJECT_ROOT = find_project_root()
OUT_DIR = PROJECT_ROOT / "outputs" / "loss_reserving"
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("seaborn-v0_8-whitegrid")


## 2. Layer 1 Data: Simulated Cumulative Paid Loss Triangle

Layer 1 uses a compact cumulative paid loss triangle created directly in the notebook. Older accident years are more mature, while newer accident years contain fewer observed development periods.


In [ ]:
# Build the Layer 1 simulated cumulative paid loss triangle directly in the
# notebook. Columns are development ages in months; rows are accident years.
# Missing values represent future development periods that have not emerged yet.

triangle = pd.DataFrame(
    {
        12: [1200, 1350, 1500, 1700, 1900],
        24: [2100, 2300, 2550, 2900, np.nan],
        36: [2600, 2800, 3150, np.nan, np.nan],
        48: [2850, 3100, np.nan, np.nan, np.nan],
        60: [3000, np.nan, np.nan, np.nan, np.nan],
    },
    index=pd.Index([2019, 2020, 2021, 2022, 2023], name="accident_year"),
).astype(float)

triangle


## 3. Layer 1 Development Factors

Loss Development Factor (LDF) = next development age cumulative paid loss / current development age cumulative paid loss.

Cumulative Development Factor (CDF) converts losses at the latest observed maturity to estimated ultimate loss.


In [ ]:
# Calculate age-to-age LDFs and CDFs to ultimate for the Layer 1 triangle.
# The resulting table is the selected development pattern used by Chain Ladder.

ldfs = calculate_ldf(triangle)
cdfs = calculate_cdf(ldfs)
development_summary = pd.concat([ldfs, cdfs], axis=1)
development_summary.round(4)


## 4. Layer 1 Chain Ladder Reserve Estimate

For each accident year: ultimate loss = latest paid loss x CDF, and IBNR = ultimate loss - latest paid loss.


In [ ]:
# Apply the Chain Ladder method to estimate ultimate loss and IBNR by accident
# year using the selected CDF for each accident year's latest maturity.

cl_result = chain_ladder_reserve(triangle)
cl_result.round(2)


## 5. Layer 1 Bornhuetter-Ferguson Reserve Estimate

The Bornhuetter-Ferguson method stabilizes reserve estimates for immature accident years by blending actual paid experience with an expected loss assumption.

Assumptions used here: expected loss ratio = 65%, and expected loss = earned premium x expected loss ratio.


In [ ]:
# Apply the Bornhuetter-Ferguson method using earned premium and a selected
# expected loss ratio. These assumptions create the expected loss estimate.

earned_premium = {2019: 4200, 2020: 4500, 2021: 4800, 2022: 5100, 2023: 5400}
summary = bornhuetter_ferguson_reserve(cl_result, earned_premium, expected_loss_ratio=0.65)
summary.round(2)


## 6. Layer 1 Reserve Summary


In [ ]:
# Keep the core Layer 1 reserve fields in a compact exhibit and export them.
# This is the type of table an analyst would use in a memo or dashboard.

reserve_summary = summary[[
    "latest_dev",
    "latest_paid",
    "cdf_to_ultimate",
    "cl_ultimate_loss",
    "cl_ibnr",
    "bf_ultimate_loss",
    "bf_ibnr",
]]

reserve_summary.to_csv(OUT_DIR / "reserve_summary.csv")
reserve_summary.round(2)


In [ ]:
# Summarize total IBNR under each method to show the overall reserve impact.
# A positive difference means Chain Ladder is higher than BF.

total_summary = pd.Series({
    "Total Chain Ladder IBNR": reserve_summary["cl_ibnr"].sum(),
    "Total BF IBNR": reserve_summary["bf_ibnr"].sum(),
    "Difference": reserve_summary["cl_ibnr"].sum() - reserve_summary["bf_ibnr"].sum(),
})

total_summary.round(2)


## 7. Layer 1 Result Diagnostics

The diagnostics below connect the reserve results back to maturity and method behavior. This is useful because unpaid claim estimates should be evaluated, not only calculated.


In [ ]:
# Build a diagnostic exhibit that explains where the reserve is coming from.
# Percent reported is 1 / CDF. Newer years should have lower percent reported
# and therefore a larger share of unpaid claims.

reserve_diagnostics = reserve_summary.copy()
reserve_diagnostics["percent_reported"] = 1 / reserve_diagnostics["cdf_to_ultimate"]
reserve_diagnostics["unreported_percent"] = 1 - reserve_diagnostics["percent_reported"]
reserve_diagnostics["cl_ibnr_share"] = (
    reserve_diagnostics["cl_ibnr"] / reserve_diagnostics["cl_ibnr"].sum()
)
reserve_diagnostics["cl_less_bf_ibnr"] = (
    reserve_diagnostics["cl_ibnr"] - reserve_diagnostics["bf_ibnr"]
)

reserve_diagnostics[[
    "latest_dev",
    "cdf_to_ultimate",
    "percent_reported",
    "unreported_percent",
    "cl_ibnr",
    "bf_ibnr",
    "cl_ibnr_share",
    "cl_less_bf_ibnr",
]].round(4)


## 8. Layer 1 Reserve Dashboard Charts


In [ ]:
# Plot latest paid loss plus Chain Ladder IBNR by accident year. The stacked
# bars visually bridge from observed paid loss to estimated ultimate loss.

fig, ax = plt.subplots(figsize=(9, 5))
reserve_summary[["latest_paid", "cl_ibnr"]].plot(kind="bar", stacked=True, ax=ax)
ax.set_title("Paid Loss and Chain Ladder IBNR by Accident Year")
ax.set_xlabel("Accident Year")
ax.set_ylabel("Loss")
ax.legend(["Latest Paid", "Chain Ladder IBNR"])
plt.tight_layout()
plt.savefig(OUT_DIR / "reserve_by_year.png", dpi=150)
plt.show()


In [ ]:
# Plot selected LDFs to review the development pattern. In a real analysis,
# unusual spikes here would be investigated before selecting final factors.

fig, ax = plt.subplots(figsize=(8, 4))
ldfs.plot(kind="line", marker="o", ax=ax)
ax.set_title("Selected Age-to-Age Development Factors")
ax.set_xlabel("Development Age")
ax.set_ylabel("Selected LDF")
plt.tight_layout()
plt.savefig(OUT_DIR / "ldf_pattern.png", dpi=150)
plt.show()


In [ ]:
# Compare ultimate loss estimates under Chain Ladder and BF. This helps show
# how much the a priori BF assumption changes the reserve indication.

fig, ax = plt.subplots(figsize=(9, 5))
reserve_summary[["cl_ultimate_loss", "bf_ultimate_loss"]].plot(kind="bar", ax=ax)
ax.set_title("Chain Ladder vs Bornhuetter-Ferguson Ultimate Loss")
ax.set_xlabel("Accident Year")
ax.set_ylabel("Ultimate Loss")
ax.legend(["Chain Ladder", "Bornhuetter-Ferguson"])
plt.tight_layout()
plt.savefig(OUT_DIR / "method_comparison.png", dpi=150)
plt.show()


## 9. Layer 2: chainladder Sample Data Analysis

This section uses the `chainladder` package as an additional data source. The RAA sample triangle is loaded directly with `cl.load_sample("RAA")`, then analyzed in two ways:

- Convert the `chainladder` Triangle to a pandas DataFrame and reuse the manual reserve functions from Layer 1.
- Fit `cl.Chainladder()` to the same sample triangle and compare package-based ultimate loss and IBNR indications.

This provides a check between transparent manual formulas and a standard actuarial Python package.


In [ ]:
# Load the built-in RAA sample triangle from chainladder for Layer 2. Convert it
# to a pandas DataFrame so the same manual helper functions can be reused.

raa_triangle_cl = cl.load_sample("RAA")
raa_triangle_df = raa_triangle_cl.to_frame()
raa_triangle_df.index = raa_triangle_df.index.year
raa_triangle_df.index.name = "accident_year"
raa_triangle_df = raa_triangle_df.astype(float)

print(type(raa_triangle_cl))
display(raa_triangle_df)


In [ ]:
# Run the manual Chain Ladder functions on the RAA sample data. This checks
# that the Layer 1 logic can also work on a standard actuarial sample triangle.

raa_ldfs = calculate_ldf(raa_triangle_df)
raa_cdfs = calculate_cdf(raa_ldfs)
raa_manual_summary = chain_ladder_reserve(raa_triangle_df)

raa_manual_summary.to_csv(OUT_DIR / "raa_manual_chain_ladder_summary.csv")

print("RAA selected development factors:")
display(raa_ldfs.round(4))

print("RAA manual Chain Ladder reserve summary:")
display(raa_manual_summary.round(2))

print("Total manual RAA IBNR:", round(raa_manual_summary["cl_ibnr"].sum(), 2))


In [ ]:
# Fit the official chainladder package model on the same RAA triangle, then
# compare package output against the manual Chain Ladder calculation.

raa_cl_model = cl.Chainladder().fit(raa_triangle_cl)

raa_package_summary = pd.DataFrame(
    {
        "chainladder_ultimate_loss": raa_cl_model.ultimate_.to_frame().iloc[:, 0],
        "chainladder_ibnr": raa_cl_model.ibnr_.to_frame().iloc[:, 0],
    }
)

# Align package output to the same accident-year index used by the manual
# summary so the two results can be joined and compared directly.
raa_package_summary.index = raa_package_summary.index.year
raa_package_summary.index.name = "accident_year"
raa_package_summary = raa_package_summary.fillna(0)
raa_package_summary.to_csv(OUT_DIR / "raa_package_chain_ladder_summary.csv")

raa_comparison = raa_manual_summary[["cl_ultimate_loss", "cl_ibnr"]].join(raa_package_summary)
raa_comparison["ultimate_difference"] = (
    raa_comparison["cl_ultimate_loss"] - raa_comparison["chainladder_ultimate_loss"]
)
raa_comparison["ibnr_difference"] = (
    raa_comparison["cl_ibnr"] - raa_comparison["chainladder_ibnr"]
)

display(raa_comparison.round(2))


In [ ]:
# Plot manual versus package-based RAA IBNR by accident year. This makes any
# differences between the transparent calculation and package model visible.

fig, ax = plt.subplots(figsize=(10, 5))
raa_comparison[["cl_ibnr", "chainladder_ibnr"]].plot(kind="bar", ax=ax)
ax.set_title("RAA Manual vs chainladder Package IBNR")
ax.set_xlabel("Accident Year")
ax.set_ylabel("IBNR")
ax.legend(["Manual Chain Ladder", "chainladder Package"])
plt.tight_layout()
plt.savefig(OUT_DIR / "raa_manual_vs_package_ibnr.png", dpi=150)
plt.show()


## 10. Actuarial Memo

### Objective
Estimate unpaid claim liabilities for a small P&C portfolio using cumulative paid loss triangles. The analysis compares a paid Chain Ladder indication with a Bornhuetter-Ferguson indication to understand how much of the reserve estimate is driven by observed paid development versus an a priori expected loss assumption.

### Data
Layer 1 uses a cumulative paid loss triangle by accident year and development age. The triangle has five accident years and development ages from 12 to 60 months. Older accident years are more mature, while newer accident years have less observed development and therefore require more projection.

Layer 2 uses the RAA sample triangle from the `chainladder` package. It is included as a secondary check that the manual calculation framework can also be applied to a standard actuarial sample triangle.

### Methodology
This review follows the basic structure described in Friedland's unpaid claim estimation text: organize claims in a development triangle, review the triangle as a diagnostic tool, select development factors, project ultimate claims, and evaluate whether the technique is appropriate for the data and environment.

The paid Chain Ladder method assumes that future paid development will be similar to prior paid development at the same maturity. In practical terms, the latest paid amount for each accident year is multiplied by the applicable cumulative development factor to estimate ultimate loss. IBNR is then calculated as ultimate loss less latest paid loss.

The Bornhuetter-Ferguson method combines actual paid emergence with expected unpaid claims. Expected loss is calculated as earned premium times the selected expected loss ratio. The unreported percentage is derived from the selected development pattern. This method gives less weight to immature paid experience and more weight to the expected loss assumption when the accident year is still early in development.

### Development Pattern
The selected paid age-to-age factors are:

- 12-24 months: 1.7130
- 24-36 months: 1.2302
- 36-48 months: 1.1019
- 48-60 months: 1.0526

The pattern is directionally reasonable for a paid loss triangle: development is strongest at early maturities and gradually tapers as claims mature. This is consistent with the diagnostic expectation that paid losses generally emerge over time and become more stable at later maturities.

The implied maturity is materially different by accident year. Accident year 2023 is only 40.9% reported on a paid basis, while accident year 2020 is approximately 95.0% reported and accident year 2019 is fully mature under the selected pattern. This maturity gradient is the main driver of the reserve distribution.

### Results
The Chain Ladder total IBNR is 4,648.67. The Bornhuetter-Ferguson total IBNR is 3,641.94. The Chain Ladder indication is therefore 1,006.73 higher than the BF indication.

The reserve is concentrated in the newest accident years. Accident year 2023 has Chain Ladder IBNR of 2,744.11, or approximately 59.0% of total Chain Ladder IBNR. Accident years 2022 and 2023 together account for approximately 85.7% of total Chain Ladder IBNR. This concentration is expected because those accident years are least mature and have the largest unreported percentages.

The mature years do not drive the reserve selection. Accident year 2019 is fully mature in the selected triangle and produces no IBNR under either method. Accident year 2020 is 48 months developed; the Chain Ladder IBNR is 163.16 and the BF IBNR is 146.25, a small difference because only about 5.0% of loss is estimated to be unreported.

The method difference becomes more meaningful for immature years. Accident year 2023 has a 12-month CDF of 2.4443, so the Chain Ladder method more than doubles the latest paid amount to estimate ultimate loss. The BF method produces a lower IBNR for 2023 because it anchors the unreported portion to expected loss rather than allowing the early paid amount to fully determine the projected ultimate.

### Evaluation of Methods
The Chain Ladder indication is most persuasive when the selected historical development pattern is credible and when current claim settlement conditions are consistent with the historical period. It is also more responsive to the actual emergence in the triangle. That responsiveness is useful when the latest experience is credible, but it can be volatile for immature years with large CDFs.

The BF indication is useful when the early paid data are not fully credible. It uses the development pattern to estimate how much loss remains unpaid, but it applies that unpaid percentage to expected loss instead of fully developing early paid loss. This makes it a useful reasonableness check for 2022 and 2023, where the observed paid data are still limited.

Based on these results, the BF indication appears more stable for the newest accident years, while the Chain Ladder indication remains an important mechanical benchmark. A selected reserve could reasonably give more consideration to BF for immature years and more consideration to Chain Ladder for mature years, subject to review of the expected loss ratio and development factor selections.

### Conclusion
The analysis indicates unpaid claims are concentrated in the most recent accident years, which is consistent with the selected paid development pattern. Chain Ladder produces the higher total IBNR because it relies more directly on early paid emergence and applies larger CDFs to immature years. BF produces a lower and more stable estimate by blending paid emergence with expected loss. For this portfolio, the final reserve judgment should focus on the credibility of early paid emergence, the support for the expected loss ratio, and the stability of the selected development pattern.
